In [0]:
%run "/Users/prdconfigsalesrep@gmail.com/databricks-project/config/env_config"

In [0]:
dbutils.widgets.dropdown("env", "dev", ["dev", "qa", "prod"])

env = dbutils.widgets.get("env")

config = get_env_config(env)

catalog = config["catalog"]
schema = config["schema"]

In [0]:

gold_path = f"/Volumes/{catalog}/{schema}/gold_volume/"

In [0]:
customers_silver = spark.sql(f"select * from {catalog}.{schema}.silver_customers")

orders_silver = spark.sql(f"select * from {catalog}.{schema}.silver_orders")

In [0]:
display(customers_silver)
display(orders_silver)

In [0]:
from pyspark.sql.functions import broadcast,col

In [0]:
orders_broadcast = broadcast(orders_silver)
customer_broadcast = broadcast(customers_silver)

In [0]:
from pyspark.sql.functions import count

gold_advanced = gold_joined.groupBy(
    "customer_id",
    "name",
    "city"
).agg(
    sum("amount").alias("total_spent"),
    count("order_id").alias("order_count")
)

In [0]:
display(gold_joined)

In [0]:
from pyspark.sql.functions import sum

In [0]:
gold_summary = gold_joined.groupBy(
    "customer_id",
    "name",
    "city"
).agg(
    sum("amount").alias("total_spent")
)

In [0]:
gold_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(gold_path)

In [0]:
gold_table = f"{catalog}.{schema}.gold_customer_spending"

gold_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(gold_table)